<a href="https://colab.research.google.com/github/Thanmai25-code/Homesafe-CQC-Kent/blob/main/Homesafe_week1_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests pandas -q
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata


In [ ]:
API_KEY= userdata.get('CQC_API_KEY')
HEADERS = {"Ocp-Apim-Subscription-Key": API_KEY}
BASE_URL = "https://api.service.cqc.org.uk/public/v1"

TARGET_REGULATED_ACTIVITY = "Personal care"
KENT_LOCAL_AUTHORITIES = ["Kent", "Medway"]
KENT_POSTCODE_PREFIXES = ("CT", "DA", "ME", "TN")

CHECKPOINT_FILE = "raw_details_checkpoint.json"
MAX_WORKERS = 5


In [ ]:
def get(url, params=None, max_retries=3):
    params = dict(params or {})
    for attempt in range(max_retries):
        resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
        if resp.status_code == 429:  # rate limited
            wait = 2 ** attempt  # 1s, 2s, 4s backoff
            print(f"Rate limited, waiting {wait}s...")
            time.sleep(wait)
            continue
        if resp.status_code != 200:
            raise RuntimeError(f"CQC API error {resp.status_code} for {url}: {resp.text[:300]}")
        return resp.json()
    raise RuntimeError(f"Gave up after {max_retries} retries on {url}")

In [ ]:
def get_all_south_east_summaries():
    summaries = []
    page = 1
    per_page = 100

    while True:
        data = get(f"{BASE_URL}/locations", params={
            "region": "South East",
            "page": page,
            "perPage": per_page,
        })
        results = data.get("locations", [])
        summaries.extend(results)

        total_pages = data.get("totalPages", 1)
        print(f"page {page}/{total_pages} -> {len(results)} locations (running total: {len(summaries)})")

        if page >= total_pages or not results:
            break
        page += 1

    return summaries

all_summaries = get_all_south_east_summaries()

# Filter locally to Kent-shaped postcodes before fetching full details
kent_candidates = [
    s for s in all_summaries
    if s.get("postalCode", "").strip().upper().startswith(KENT_POSTCODE_PREFIXES)
]
all_ids = {s["locationId"] for s in kent_candidates if "locationId" in s}
print(f"\n{len(kent_candidates)} candidates with Kent-shaped postcodes, {len(all_ids)} unique IDs.")

page 1/216 -> 100 locations (running total: 100)
page 2/216 -> 100 locations (running total: 200)
page 3/216 -> 100 locations (running total: 300)
page 4/216 -> 100 locations (running total: 400)
page 5/216 -> 100 locations (running total: 500)
page 6/216 -> 100 locations (running total: 600)
page 7/216 -> 100 locations (running total: 700)
page 8/216 -> 100 locations (running total: 800)
page 9/216 -> 100 locations (running total: 900)
page 10/216 -> 100 locations (running total: 1000)
page 11/216 -> 100 locations (running total: 1100)
page 12/216 -> 100 locations (running total: 1200)
page 13/216 -> 100 locations (running total: 1300)
page 14/216 -> 100 locations (running total: 1400)
page 15/216 -> 100 locations (running total: 1500)
page 16/216 -> 100 locations (running total: 1600)
page 17/216 -> 100 locations (running total: 1700)
page 18/216 -> 100 locations (running total: 1800)
page 19/216 -> 100 locations (running total: 1900)
page 20/216 -> 100 locations (running total: 2000

In [ ]:
def fetch_location_detail(location_id):
    return get(f"{BASE_URL}/locations/{location_id}")

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        raw_details = json.load(f)
    print(f"Resumed from checkpoint: {len(raw_details)} already fetched.")
else:
    raw_details = {}

all_ids_list = sorted(all_ids)
remaining = [lid for lid in all_ids_list if lid not in raw_details]
print(f"{len(remaining)} left to fetch, using {MAX_WORKERS} parallel workers.")

def fetch_one(location_id):
    try:
        return location_id, fetch_location_detail(location_id), None
    except RuntimeError as e:
        return location_id, None, str(e)

completed = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(fetch_one, lid): lid for lid in remaining}
    for future in as_completed(futures):
        location_id, detail, error = future.result()
        completed += 1
        if error:
            print(f"SKIPPED {location_id}: {error}")
        else:
            raw_details[location_id] = detail

        if completed % 100 == 0:
            with open(CHECKPOINT_FILE, "w") as f:
                json.dump(raw_details, f)
            print(f"[{completed}/{len(remaining)}] checkpoint saved — {len(raw_details)} total so far")

with open(CHECKPOINT_FILE, "w") as f:
    json.dump(raw_details, f)
print(f"\nDone. {len(raw_details)} full records fetched.")


Resumed from checkpoint: 5137 already fetched.
0 left to fetch, using 5 parallel workers.

Done. 5137 full records fetched.


In [ ]:
def is_personal_care(detail):
    activities = detail.get("regulatedActivities", []) or []
    has_personal_care = any((a.get("name") or "").strip().lower() == TARGET_REGULATED_ACTIVITY.lower()
                             for a in activities)
    in_kent = detail.get("localAuthority") in KENT_LOCAL_AUTHORITIES
    return has_personal_care and in_kent

def flatten_location(detail):
    current_rating = detail.get("currentRatings", {}) or {}
    overall = current_rating.get("overall", {}) or {}
    activities = [a.get("name") for a in (detail.get("regulatedActivities") or [])]
    service_types = [s.get("name") for s in (detail.get("gacServiceTypes") or [])]

    return {
        "locationId": detail.get("locationId"),
        "providerId": detail.get("providerId"),
        "locationName": detail.get("name"),
        "registrationStatus": detail.get("registrationStatus"),
        "registrationDate": detail.get("registrationDate"),
        "addressLine1": detail.get("postalAddressLine1"),
        "town": detail.get("postalAddressTownCity"),
        "postcode": detail.get("postalCode"),
        "region": detail.get("region"),
        "localAuthority": detail.get("localAuthority"),
        "constituency": detail.get("constituency"),
        "latitude": detail.get("onspdLatitude"),
        "longitude": detail.get("onspdLongitude"),
        "phoneNumber": detail.get("mainPhoneNumber"),
        "regulatedActivities": "; ".join(filter(None, activities)),
        "gacServiceTypes": "; ".join(filter(None, service_types)),
        "overallRating": overall.get("rating"),
        "overallRatingDate": overall.get("reportDate"),
        "numberOfBeds": detail.get("numberOfBeds"),
        "website": detail.get("website"),
    }

rows = [flatten_location(d) for d in raw_details.values() if is_personal_care(d)]
print(f"{len(rows)} personal care locations kept out of {len(raw_details)} checked.")

579 personal care locations kept out of 5137 checked.


In [ ]:
df = pd.DataFrame(rows)
df.to_csv('kent_personal_care_locations.csv', index=False)
print(df['localAuthority'].value_counts())
df.head()

localAuthority
Kent      453
Medway    126
Name: count, dtype: int64


,locationId,providerId,locationName,registrationStatus,registrationDate,addressLine1,town,postcode,region,localAuthority,constituency,latitude,longitude,phoneNumber,regulatedActivities,gacServiceTypes,overallRating,overallRatingDate,numberOfBeds,website
0,1-10064258273,1-9491037344,Kent Case Management Ltd,Registered,2020-12-30,"The Stables, Bradbourne House,",West Malling,ME19 6DZ,South East,Kent,Maidstone and Malling,51.295547,0.441755,07741497668,Personal care,Homecare agencies,Good,2022-03-18,0.0,www.kentcasemanagement.co.uk
1,1-10086956930,1-8974776775,Assured Healthcare Limited,Registered,2020-12-24,75 Wiltshire Close,Chatham,ME5 7SS,South East,Medway,Chatham and Aylesford,51.365265,0.547372,01634320013,"Personal care; Treatment of disease, disorder ...",Community services - Healthcare; Homecare agen...,None,None,0.0,www.assuredhc.co.uk
2,1-10087632956,1-6192052130,Royalcare- Thanet,Registered,2020-12-23,Kent Innovation Centre,Broadstairs,CT10 2QQ,South East,Kent,East Thanet,51.358932,1.406302,01843838122,Personal care,Homecare agencies,Requires improvement,2022-05-25,0.0,www.royalcare24.co.uk
3,1-10204301199,1-118164017,Priory Supported Living Kent,Registered,2021-02-25,Buckland,Dover,CT17 0TQ,South East,Kent,Dover and Deal,51.135385,1.298625,01304202010,Personal care,Homecare agencies; Supported living,Good,2022-11-11,0.0,www.prioryadultcare.co.uk
4,1-10200463066,1-3982604534,PCAS Kent Ltd,Registered,2021-01-14,Unit 5,Faversham,ME13 8GD,South East,Kent,Faversham and Mid Kent,51.310679,0.898310,03300535919,Personal care,Homecare agencies,None,None,0.0,None


In [ ]:
print(f"Total rows: {len(df)}")

Total rows: 579
registrationStatus
Registered    579
Name: count, dtype: int64
0
510


In [ ]:
from google.colab import files
files.download('kent_personal_care_locations.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>